# Hello World: Quantum Machine Learning with Merlin (Cloud)

Welcome! This notebook demonstrates how to use Merlin to build a quantum reservoir algorithm to then run on a real QPU (or in this case, a simulator emulating it). We chose a reservoir algorithm since gradient is not propagated through quantum layers when using a processor.

## 1. Install and Import Dependencies

First, lets make sure all required packages are installed and import them.  
If you haven't installed Merlin yet, run:  
`pip install merlinquantum` in your terminal or `!pip install merlinquantum` in your notebook.

In [ ]:
import torch
import torch.nn as nn
import merlin as ML
import perceval as pcvl
from merlin.datasets import iris

To run your experiments on a processor (real QPU or a simulator emulating it), we need to create a `MerlinProcessor` object.

In [ ]:
#(OPTIONAL) Save your token in a file called .env in the same directory as this notebook, with the following content:
# CLOUD_TOKEN=your_token_here
from dotenv import load_dotenv
load_dotenv()
import os
CLOUD_TOKEN = os.getenv("CLOUD_TOKEN")

Lets first load a perceval `RemoteProcessor`. It is the original way of accessing Quandela's cloud. For Scaleway hosted plateforms and any future session-based providers, use `pcvl.providers.scaleway` instead of `pcvl.RemoteProcessor`.

Here we use `sim:slos` which is a noise-free simulator, use `sim:ascella` if you want a simultor which reproduces the noise of the `qpu:ascella` QPU.

In [ ]:
pcvl.RemoteConfig.set_token(CLOUD_TOKEN)
rp = pcvl.RemoteProcessor("sim:slos")

In [ ]:
proc = ML.MerlinProcessor(
    rp,
    microbatch_size=32,        # batch chunk size per cloud call
    timeout=3600.0,            # default wall-time per forward (seconds)
    max_shots_per_call=None,   # optional cap per cloud call (see below)
    chunk_concurrency=1,       # parallel chunk jobs within a quantum leaf
)

## 2. Load and Prepare the Iris Dataset

We'll use the classic Iris dataset, a simple and well-known benchmark for classification.  
Let's load the data and convert it to PyTorch tensors for training.

In [ ]:
train_features, train_labels, train_metadata = iris.get_data_train()
test_features, test_labels, test_metadata = iris.get_data_test()

# Convert data to PyTorch tensors
X_train = torch.FloatTensor(train_features)
y_train = torch.LongTensor(train_labels)
X_test = torch.FloatTensor(test_features)
y_test = torch.LongTensor(test_labels)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]}")
print(f"Classes: {len(torch.unique(y_train))}")

![iris](../img/iris.png)

## 3. Define the Quantum reservoir Model

The model can be split into two parts:
- The `QuantumLayer` implements the quantum reservoir.
- The `classical_out` is the classical model that takes the reservoir's output to classify the data.

To make sure that the model runs on the processor, we will need to call the processor's `forward` method. We will also need to redefine the `parameters` method so that only the classical parameters are changed and not the reservoir's (MerLin's simple quantum layer creates trainable pytorch parameter by default).

In [ ]:
class HybridIrisClassifier(nn.Module):
    """
    Hybrid model for Iris classification:
    - Quantum reservoir processes the 4 features
    - Classical output layer for 3-class classification
    """
    def __init__(self):
        super(HybridIrisClassifier, self).__init__()

        # Quantum layer: processes the 4 features
        self.quantum = ML.QuantumLayer.simple(
            input_size=4,
        ).eval()
        # Classical output layer: quantum → 8 → 3
        self.classical_out = nn.Sequential(
            nn.Linear(self.quantum.output_size, 8),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(8, 3)
        )
        self.params=self.classical_out.parameters()

    def forward(self, x):
        #return self.classical_out(proc.forward(self.quantum.eval(),x))
        #TODO Use the commented return if you want to use the reservoir algorithm without the processor.
        return self.classical_out(self.quantum.eval()(x))
    
    def parameters(self):
        return self.params

This could be ran easily with the processor, but it takes a lot of time. Since the reservoir always return the same output (it is not trained and receives the same input), we can calculate all of the outputs of the reservoir and then reuse them. The next two code cells implement the same reservoir as earlier but in a more resource-efficient way.

Lets calculate all of the reservoir outputs and add them in a dictionary.

In [ ]:
reservoir=ML.QuantumLayer.simple(
            input_size=4,
        ).eval()

output_size=reservoir.output_size

train_outputs= proc.forward(reservoir,X_train)
train_reservoir_map={tuple(x.tolist()):output for x,output in zip(X_train,train_outputs)}
test_outputs=proc.forward(reservoir,X_test)
test_reservoir_map={tuple(x.tolist()):output for x,output in zip(X_test,test_outputs)}

reservoir_map = {**train_reservoir_map, **test_reservoir_map}

In [ ]:
reservoir_map


We can easily define the trainable classical model using this map.

In [ ]:
class HybridIrisClassifier(nn.Module):
    """
    Hybrid model for Iris classification:
    - Quantum reservoir processes the 4 features
    - Classical output layer for 3-class classification
    """
    def __init__(self,output_size:int=1):
        super(HybridIrisClassifier, self).__init__()
        self.output_size=output_size

        # Classical output layer: quantum → 8 → 3
        self.model = nn.Sequential(
            nn.Linear(output_size, 8),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(8, 3)
        )
        

    def forward(self, x:torch.Tensor):
        if x.dim()==1:
            x.unsqueeze(0)
        input_to_classical=torch.empty(x.shape[0],self.output_size)
        for i,input in enumerate(x):
            input_to_classical[i]=reservoir_map[tuple(input.tolist())]

        return self.model(input_to_classical)

## 4. Set the Training Parameters

You can adjust these parameters to see how they affect training and model performance.

In [ ]:
learning_rate = 0.01
number_of_epochs = 200

## 5. Train the Hybrid Model

Lets train the model like a normal pytorch module.

In [ ]:
import random, numpy as np, torch
def reset_seeds(s=0):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

reset_seeds(123)
model = HybridIrisClassifier(output_size=output_size)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()
model.train()

#Training loop
for epoch in range(number_of_epochs):
    optimizer.zero_grad()
    loss = criterion(model(X_train), y_train)
    loss.backward()
    optimizer.step()
    model.eval()
    with torch.no_grad():
        preds = model(X_test).argmax(dim=1)
        accuracy=(preds == y_test).float().mean().item()
    model.train()
    print(f"Epoch {epoch+1} had a loss of {loss.item()} and a test accuracy of {accuracy}")

## 6. Evaluate the Model

After training, let's evaluate our model on the test set and print the accuracy.

In [ ]:
# Evaluate on test set
model.eval()
with torch.no_grad():
    test_outputs = model(X_test)
    predictions = torch.argmax(test_outputs, dim=1)
    accuracy = (predictions == y_test).float().mean().item()
    print(f"Test accuracy: {accuracy:.4f}")

In [ ]:
number_of_runs = 10
accuracies = []

for i in range(number_of_runs):
    reset_seeds(i+123)
    model = HybridIrisClassifier(output_size=output_size)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    #Training loop 
    model.train()
    for epoch in range(number_of_epochs):
        optimizer.zero_grad()
        loss = criterion(model(X_train), y_train)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            preds = model(X_test).argmax(dim=1)
            accuracy=(preds == y_test).float().mean().item()
        model.train()
        #print(f"Epoch {epoch+1} had a loss of {loss.item()} and a test accuracy of {accuracy}")

    #Final evaluation of the model
    model.eval()
    with torch.no_grad():
        preds = model(X_test).argmax(dim=1)
        accuracies.append((preds == y_test).float().mean().item())

avg = torch.tensor(accuracies).mean().item()
std = torch.tensor(accuracies).std(unbiased=True).item()
print(f"Average accuracy: {avg:.4f} ± {std:.4f}")


# Conclusion

Congratulations! You've trained and evaluated a hybrid quantum-classical neural network using Merlin.  
Feel free to experiment with the model architecture, quantum parameters, or try other datasets!

Even though MerLin is built as a simulation-first package, it is still possible to run and optimize quantum layers with the processor. Although, a gradient-free optimizer such as COBYLA should be used.

Also, if you want a better performing reservoir based on the litterature, you can try to reproduce the [Quantum optical reservoir computing powered by boson sampling](https://opg.optica.org/opticaq/fulltext.cfm?uri=opticaq-3-3-238&id=572317)'s resevoir. It is good exercice to familiarize yourself with MerLin and learn about PCA if you are not from a machine learning background.